In [3]:
# ============================================================
# RetailGPT - Enterprise Data Generator
# ============================================================

import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

np.random.seed(42)
random.seed(42)

print("Libraries Loaded")

Libraries Loaded


In [4]:
retailers = [

"DMart",

"Reliance Fresh",

"Big Bazaar",

"More",

"Spencer's",

"Star Bazaar",

"Metro Cash & Carry",

"Easyday",

"Nature's Basket",

"Vishal Mega Mart"

]

regions = [

"North",

"South",

"East",

"West"

]

states = [

"Karnataka",

"Tamil Nadu",

"Maharashtra",

"Delhi",

"UP",

"Rajasthan",

"Kerala",

"Telangana"

]

channels = [

"Modern Trade",

"General Trade",

"E-Commerce"

]

In [5]:
products = [

("Pringles Original","Pringles","Snacks"),

("Pringles Sour Cream","Pringles","Snacks"),

("Pringles Hot & Spicy","Pringles","Snacks"),

("Chocos","Kellogg's","Cereal"),

("Corn Flakes","Kellogg's","Cereal"),

("Muesli","Kellogg's","Cereal"),

("Oats","Kellogg's","Breakfast"),

("Granola","Kellogg's","Breakfast")

]

In [6]:
sku_master = []

for i,p in enumerate(products):

    sku_master.append({

        "SKU_ID":1000+i,

        "SKU_NAME":p[0],

        "BRAND":p[1],

        "CATEGORY":p[2],

        "PACK_SIZE":random.choice([50,100,150,200,250]),

        "MRP":random.choice([10,20,30,50,99,120,150])

    })

sku_master=pd.DataFrame(sku_master)

sku_master.head()

,SKU_ID,SKU_NAME,BRAND,CATEGORY,PACK_SIZE,MRP
0,1000,Pringles Original,Pringles,Snacks,50,10
1,1001,Pringles Sour Cream,Pringles,Snacks,150,20
2,1002,Pringles Hot & Spicy,Pringles,Snacks,100,20
3,1003,Chocos,Kellogg's,Cereal,50,120
4,1004,Corn Flakes,Kellogg's,Cereal,250,10


In [7]:
retailer_master=[]

for i in range(100):

    retailer_master.append({

        "STORE_ID":i+1,

        "STORE_NAME":fake.company(),

        "RETAILER":random.choice(retailers),

        "STATE":random.choice(states),

        "REGION":random.choice(regions),

        "CHANNEL":random.choice(channels)

    })

retailer_master=pd.DataFrame(retailer_master)

retailer_master.head()

,STORE_ID,STORE_NAME,RETAILER,STATE,REGION,CHANNEL
0,1,Young-Holloway,More,Karnataka,South,E-Commerce
1,2,"Bautista, Chavez and Thomas",Nature's Basket,Kerala,South,General Trade
2,3,Fernandez-Acosta,Vishal Mega Mart,UP,North,Modern Trade
3,4,Barry-Lopez,Metro Cash & Carry,Rajasthan,East,Modern Trade
4,5,"Duncan, Ballard and Porter",More,Rajasthan,North,Modern Trade


In [8]:
sales=[]

dates=pd.date_range(

"2024-01-01",

"2025-12-31"

)

for day in dates:

    for _ in range(100):

        sku=sku_master.sample(1).iloc[0]

        store=retailer_master.sample(1).iloc[0]

        qty=np.random.randint(5,100)

        sales.append({

            "DATE":day,

            "STORE_ID":store.STORE_ID,

            "SKU_ID":sku.SKU_ID,

            "SALES_VALUE":qty*sku.MRP,

            "QUANTITY":qty,

            "PROMOTION":random.choice([0,1])

        })

sales=pd.DataFrame(sales)

sales.head()

,DATE,STORE_ID,SKU_ID,SALES_VALUE,QUANTITY,PROMOTION
0,2024-01-01,77,1001,740,37,1
1,2024-01-01,16,1006,480,48,1
2,2024-01-01,15,1007,620,31,1
3,2024-01-01,41,1005,1700,34,1
4,2024-01-01,46,1006,280,28,0


In [9]:
sales=sales.merge(

sku_master,

on="SKU_ID"

)

sales=sales.merge(

retailer_master,

on="STORE_ID"

)

sales.head()

,DATE,STORE_ID,SKU_ID,SALES_VALUE,QUANTITY,PROMOTION,SKU_NAME,BRAND,CATEGORY,PACK_SIZE,MRP,STORE_NAME,RETAILER,STATE,REGION,CHANNEL
0,2024-01-01,77,1001,740,37,1,Pringles Sour Cream,Pringles,Snacks,150,20,"Beck, Powers and Andrade",Reliance Fresh,Maharashtra,East,Modern Trade
1,2024-01-01,16,1006,480,48,1,Oats,Kellogg's,Breakfast,50,10,"Lewis, Garcia and Jones",Metro Cash & Carry,UP,South,E-Commerce
2,2024-01-01,15,1007,620,31,1,Granola,Kellogg's,Breakfast,50,20,Jacobson-Burnett,Big Bazaar,Delhi,South,General Trade
3,2024-01-01,41,1005,1700,34,1,Muesli,Kellogg's,Cereal,250,50,"Nelson, Cochran and Mitchell",Star Bazaar,Telangana,West,Modern Trade
4,2024-01-01,46,1006,280,28,0,Oats,Kellogg's,Breakfast,50,10,Howard LLC,More,UP,West,Modern Trade


In [10]:
print("Sales Rows :",len(sales))

print("Stores :",sales.STORE_ID.nunique())

print("SKUs :",sales.SKU_ID.nunique())

print("Revenue :",sales.SALES_VALUE.sum())

Sales Rows : 73100
Stores : 100
SKUs : 8
Revenue : 124352910


In [11]:
import os

os.makedirs("data",exist_ok=True)

sales.to_csv("data/sales.csv",index=False)

sku_master.to_csv("data/sku_master.csv",index=False)

retailer_master.to_csv("data/retailer_master.csv",index=False)

print("Files Saved")

Files Saved


In [20]:
import sqlite3
from pathlib import Path

DATA_FOLDER = Path("data")      # use "data" if notebook isn't inside notebooks/
DB_PATH = Path("../commercial.db")

In [21]:
sales_df = pd.read_csv(DATA_FOLDER / "sales.csv")

In [24]:

conn = sqlite3.connect(DB_PATH)

csv_files = list(DATA_FOLDER.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files\n")

for csv in csv_files:

    df = pd.read_csv(csv)

    table_name = csv.stem.lower()

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"✓ Created table : {table_name} ({len(df)} rows)")

conn.commit()
conn.close()

print("\nCommercial Database Created Successfully")
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
""",
conn
)

tables

Found 3 CSV files

✓ Created table : sales (73100 rows)
✓ Created table : sku_master (8 rows)
✓ Created table : retailer_master (100 rows)

Commercial Database Created Successfully


,name
0,sales
1,sku_master
2,retailer_master


In [25]:
pd.read_sql(
"""
SELECT *
FROM sales
LIMIT 5
""",
conn
)

,DATE,STORE_ID,SKU_ID,SALES_VALUE,QUANTITY,PROMOTION,SKU_NAME,BRAND,CATEGORY,PACK_SIZE,MRP,STORE_NAME,RETAILER,STATE,REGION,CHANNEL
0,2024-01-01,77,1001,740,37,1,Pringles Sour Cream,Pringles,Snacks,150,20,"Beck, Powers and Andrade",Reliance Fresh,Maharashtra,East,Modern Trade
1,2024-01-01,16,1006,480,48,1,Oats,Kellogg's,Breakfast,50,10,"Lewis, Garcia and Jones",Metro Cash & Carry,UP,South,E-Commerce
2,2024-01-01,15,1007,620,31,1,Granola,Kellogg's,Breakfast,50,20,Jacobson-Burnett,Big Bazaar,Delhi,South,General Trade
3,2024-01-01,41,1005,1700,34,1,Muesli,Kellogg's,Cereal,250,50,"Nelson, Cochran and Mitchell",Star Bazaar,Telangana,West,Modern Trade
4,2024-01-01,46,1006,280,28,0,Oats,Kellogg's,Breakfast,50,10,Howard LLC,More,UP,West,Modern Trade
